<a href="https://colab.research.google.com/github/dkgoal/learning-R/blob/master/guru-stock-screener-3gn684/notebooks/guru_screener_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guru Stock Screener — run on Google Colab

Runs the screener on Colab's Linux VM and shows the web UI **right inside this notebook**. Works from any browser, including **iPad Safari** — no local install, and (unlike an iPad on its own) Colab can pull live SEC data.

**How to use:** `Runtime → Run all`, or run each cell top to bottom. The app appears at the bottom.

Personal use only — not investment advice.

## 1. Get the code and install dependencies

In [ ]:
!git clone https://github.com/dkgoal/learning-R.git
%cd learning-R
!git checkout claude/guru-stock-screener-3gn684
!pip install -q -r requirements.txt

## 2. (Optional) Persist data to Google Drive

Colab VMs are wiped when the runtime disconnects (~90 min idle), which clears the local cache. **Skip this for demo data.** For live data you want to keep, uncomment and run this cell **before** the next one — it stores config + cache in your Drive via the `GURU_HOME` variable the app already honors.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["GURU_HOME"] = "/content/drive/MyDrive/guru-screener"
print("Config + cache will persist at:", os.environ["GURU_HOME"])

## 3. Load data

**Demo** data is instant and needs no network. For **real** data, first edit `config/settings.yaml` — set `app.sec_user_agent` to your name + email (SEC requires it) — then use the `refresh` line instead of `seed`.

In [ ]:
!python -m guru_screener seed
# !python -m guru_screener refresh     # live SEC filings + prices (slower)

## 4. Launch the app

Starts the web server in the background and embeds the UI below. Colab proxies the port to your browser, so only **you** (in your authenticated Colab session) can reach it — safe without a login. On iPad the inline frame is the most reliable; the `serve_kernel_port_as_window` line opens it in a full tab if your browser allows popups.

In [ ]:
import threading, time, socket
from guru_screener.web.app import create_app
from guru_screener.config import load_config

def _free_port():
    s = socket.socket(); s.bind(("127.0.0.1", 0))
    p = s.getsockname()[1]; s.close(); return p

# Safe to re-run: reuse the server started earlier instead of binding twice.
# (A second bind on the same port is what raises "Address already in use".)
if not globals().get("_GURU_PORT"):
    _GURU_PORT = _free_port()
    _app = create_app(load_config())
    threading.Thread(
        target=lambda: _app.run(host="127.0.0.1", port=_GURU_PORT,
                                use_reloader=False),
        daemon=True,
    ).start()
    time.sleep(2)  # give the server a moment to start

from google.colab import output
from google.colab.output import eval_js
from IPython.display import display, HTML

# A tappable link — reliable on iPad (Safari blocks scripted pop-ups, but a real
# tap opens fine). Tap it to open the app in a full tab.
_url = eval_js(f"google.colab.kernel.proxyPort({_GURU_PORT})")
display(HTML(
    f'<p style="font-size:18px"><a href="{_url}" target="_blank" '
    f'rel="noopener">▶ Open Guru Stock Screener in a new tab</a></p>'))

# …and also embed it inline below (scroll down; may render blank on some iPads).
output.serve_kernel_port_as_iframe(_GURU_PORT, path="/", height="800")

## 5. Open on iPad / any device (recommended)

The inline frame above works within Colab, but opening Colab's own proxy link in
a separate tab trips **iPad Safari's cross-site-cookie protection** and shows a
blank page. The cell below opens a **Cloudflare quick tunnel** and prints a
normal `https://….trycloudflare.com` link that opens fine anywhere — tap it.

⚠ That link is **public** (random and unguessable) and the app has **no login**,
so treat it as temporary. It stops when you stop the cell or the runtime — don't
share it.

In [ ]:
# Public link that works reliably on iPad. (Colab's own proxy trips Safari's
# cross-site-cookie protection and can show a blank page.) This opens a
# Cloudflare quick tunnel to the same local server and prints a normal
# https://....trycloudflare.com URL — tap it to open the app on any device.
import os, re, subprocess, time, urllib.request
from IPython.display import display, HTML

if not os.path.exists("/tmp/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)

_tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{_GURU_PORT}",
     "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

_url, _t0 = None, time.time()
for _line in _tunnel.stdout:
    _m = re.search(r"https://[-\w.]+\.trycloudflare\.com", _line)
    if _m:
        _url = _m.group(0)
        break
    if time.time() - _t0 > 60:
        break

if _url:
    display(HTML(
        f'<p style="font-size:18px">📈 <a href="{_url}" target="_blank" '
        f'rel="noopener">Open Guru Stock Screener</a></p>'
        f'<p style="color:#666">{_url}</p>'))
else:
    print("Tunnel didn't come up in time — just re-run this cell.")

---
Tip: after the app loads, use the **Screener**, **Managers**, and **Backtest** tabs at the top. The backtest is deliberately labelled *approximate — not point-in-time*.